# Self-RePrompt LoRA 推理交互测试

**用途**：加载任意一个训练好的 LoRA adapter，输入自定义问题，观察模型是否输出 `<SRP_START>…<SRP_END>` 格式。

**快速开始**：
1. 修改 Cell 1 中的 `ADAPTER_DIR`
2. 依次运行所有 Cell
3. 在最后一个 Cell 输入你的问题

In [ ]:
# ── Cell 1：配置路径和设备 ────────────────────────────────────────
BASE_MODEL  = "../model/Qwen3-8B-Base"                 # 须与训练 LoRA 时基座一致（默认 Base）
ADAPTER_DIR = "../outputs/qwen3_sr_lora_v3_base"       # LoRA adapter 路径
DEVICE      = "cuda:0"                                 # 使用的 GPU
MAX_NEW_TOKENS = 512                                   # 最大生成 token 数

In [ ]:
# ── Cell 2：加载模型（只需运行一次）──────────────────────────────
import sys, os, json, textwrap
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 确保项目根目录在 path 里
ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

SRP_START = "<SRP_START>"
SRP_END   = "<SRP_END>"

print("[1/3] 加载 tokenizer ...")
# 从 base 加载，再添加 adapter 的特殊 token（避免 adapter tokenizer_config 格式兼容问题）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
adapter_tok_cfg = Path(ADAPTER_DIR) / "tokenizer_config.json"
if adapter_tok_cfg.exists():
    cfg = json.loads(adapter_tok_cfg.read_text())
    extra = cfg.get("extra_special_tokens", [])
    if extra:
        tokenizer.add_special_tokens({"additional_special_tokens": extra})
print(f"      词表大小: {len(tokenizer)}  "
      f"SRP_START id={tokenizer.convert_tokens_to_ids(SRP_START)}  "
      f"SRP_END id={tokenizer.convert_tokens_to_ids(SRP_END)}")

print("[2/3] 加载 base model ...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16,
    device_map=DEVICE, trust_remote_code=True,
)
base.resize_token_embeddings(len(tokenizer))  # 对齐 adapter 词表大小

print("[3/3] 挂载 LoRA adapter ...")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("✅ 模型加载完成，可以开始推理")

[1/3] 加载 tokenizer ...
      词表大小: 151671  SRP_START id=151669  SRP_END id=151670
[2/3] 加载 base model ...


RuntimeError: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 802: system not yet initialized

In [3]:
# ── Cell 3：推理函数定义 ──────────────────────────────────────────
def infer(user_input: str, max_new_tokens: int = MAX_NEW_TOKENS,
          do_sample: bool = False, temperature: float = 0.7,
          show_raw: bool = False) -> str:
    """
    对单条 user 输入进行推理，打印格式化结果。

    参数
    ----
    user_input     : 用户问题文本
    max_new_tokens : 最大生成 token 数
    do_sample      : 是否采样（True=随机，False=greedy）
    temperature    : 采样温度（do_sample=True 时生效）
    show_raw       : 是否额外打印原始 token 序列（调试用）
    """
    msgs   = [{"role": "user", "content": user_input}]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    new_ids  = out[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=False)

    # ── 解析 SRP 段 ──
    has_start = SRP_START in response
    has_end   = SRP_END   in response
    srp_ok    = has_start and has_end and response.index(SRP_START) < response.index(SRP_END)

    srp_content  = ""
    main_content = response
    if srp_ok:
        s = response.index(SRP_START) + len(SRP_START)
        e = response.index(SRP_END)
        srp_content  = response[s:e].strip()
        main_content = response[response.index(SRP_END) + len(SRP_END):].strip()
        # 去掉结尾的 <|im_end|>
        main_content = main_content.replace("<|im_end|>", "").strip()

    # ── 打印 ──
    sep = "─" * 68
    print(sep)
    print(f"📥 User: {user_input}")
    print(sep)
    if srp_ok:
        print(f"🧠 [SRP 策略段]")
        print(f"   {srp_content}")
        print()
        print(f"💬 [最终回答]")
        # 自动换行显示
        for line in main_content.split("\n"):
            print(textwrap.fill(line, width=80, subsequent_indent="   ") if line.strip() else "")
    else:
        print(f"⚠️  未检测到 SRP 格式 (SRP_START={has_start}, SRP_END={has_end})")
        print(response)
    print(sep)

    if show_raw:
        print("[raw response]")
        print(repr(response))

    return response

def infer_compare(user_input: str, max_new_tokens: int = MAX_NEW_TOKENS) -> tuple:
    """同时展示 LoRA 微调后与原始 base 的回复，便于对比。"""
    def _gen(use_lora: bool) -> str:
        if use_lora:
            model.enable_adapter_layers()
            msgs = [{"role": "user", "content": user_input}]
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            model.disable_adapter_layers()
            msgs = [{"role": "user", "content": user_input}]
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False) #拒绝 thinking
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)

    resp_lora = _gen(True)
    resp_base = _gen(False)
    model.enable_adapter_layers()  # 恢复 LoRA

    def _parse(s: str):
        has_s = SRP_START in s and SRP_END in s and s.index(SRP_START) < s.index(SRP_END)
        srp, main = "", s
        if has_s:
            srp = s[s.index(SRP_START)+len(SRP_START):s.index(SRP_END)].strip()
            main = s[s.index(SRP_END)+len(SRP_END):].replace("<|im_end|>", "").strip()
        return has_s, srp, main

    sep = "─" * 68
    print(sep)
    print(f"📥 User: {user_input}")
    print(sep)
    for label, resp in [("🔷 Self-RePrompt", resp_lora), ("🔶 Base", resp_base)]:
        ok, srp, main = _parse(resp)
        print(f"\n{label}")
        if ok:
            print(f"   [SRP] {srp}")
            for line in main.split("\n"):
                print(textwrap.fill(line, width=76, initial_indent="   ", subsequent_indent="   ") if line.strip() else "   ")
        else:
            txt = resp.replace("<|im_end|>", "").strip()
            for line in txt.split("\n"):
                print(textwrap.fill(line, width=76, initial_indent="   ", subsequent_indent="   ") if line.strip() else "   ")
    print(sep)
    return resp_lora, resp_base

print("✅ infer() / infer_compare() 已定义")

✅ infer() / infer_compare() 已定义


In [3]:
# ── Cell 4：快速冒烟测试（预置例题）─────────────────────────────
# EXAMPLES = [
#     # HotpotQA 风格
#     "Were Scott Derrickson and Ed Wood from the same country?",
#     # GSM8K 风格
#     "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
#     # 开放域
#     "What is the difference between supervised and unsupervised learning?",
# ]
EXAMPLES =["What can you do for me?",
            "What nationality was James Henry Miller's wife?"
]
EXAMPLES =["Describe a time when you had to make a difficult decision."
]

EXAMPLES =["The sun is responsible for? A: puppies learning new tricks. B: children growing up and getting old. C: flowers wilting in a vase. D: plants sprouting, blooming and wilting."
]

EXAMPLES =["ypical advertising regulatory bodies suggest, for example that adverts must not: encourage _________, cause unnecessary ________ or _____, and must not cause _______ offence. options:"
        "Safe practices, Fear, Jealousy, Trivial",
        "Unsafe practices, Distress, Joy, Trivial",
        "Safe practices, Wants, Jealousy, Trivial",
        "Safe practices, Distress, Fear, Trivial",
        "Unsafe practices, Wants, Jealousy, Serious",
        "Safe practices, Distress, Jealousy, Serious",
        "Safe practices, Wants, Fear, Serious",
        "Unsafe practices, Wants, Fear, Trivial",
        "Unsafe practices, Distress, Fear, Serious"
]
# EXAMPLES =["What caused the 2008 financial crisis?"
# ]

for q in EXAMPLES:
    infer_compare(q)
    print()



NameError: name 'infer_compare' is not defined

In [5]:
# ── Cell 5：自定义输入（修改这里的字符串后运行即可）────────────
MY_QUESTION = """
请在这里输入你的问题
""".strip()

infer_compare(MY_QUESTION, max_new_tokens=512)

────────────────────────────────────────────────────────────────────
📥 User: 请在这里输入你的问题
────────────────────────────────────────────────────────────────────
⚠️  未检测到 SRP 格式 (SRP_START=True, SRP_END=False)
<think>

</think>

<SRP_START>
Of course, I'm here to help. Could you please share the question you'd like assistance with?<|im_end|>
────────────────────────────────────────────────────────────────────


"<think>\n\n</think>\n\n<SRP_START>\nOf course, I'm here to help. Could you please share the question you'd like assistance with?<|im_end|>"

In [6]:
# ── Cell 6：批量测试（从文件读取）────────────────────────────────
import json

DATA_FILE   = "../data/srp_prompt_with_answer/hotpot_train_qa_2000_with_srp_answer.jsonl"
N_SAMPLES   = 5    # 测试前 N 条
SHOW_GOLD   = True # 是否打印 gold answer

samples = []
with open(DATA_FILE) as f:
    for i, line in enumerate(f):
        if i >= N_SAMPLES: break
        samples.append(json.loads(line))

for s in samples:
    infer(s["user"])
    if SHOW_GOLD:
        print(f"  📌 Gold sr_prompt : {s.get('sr_prompt', 'N/A')}")
        print(f"  📌 Gold answer    : {s.get('srp_answer', s.get('answer', 'N/A'))[:200]}")
    print()

KeyboardInterrupt: 

In [ ]:
# ── Cell 7：对比有无 LoRA 的输出（需提前加载好 model）──────────
# 用同一个 base model 在 LoRA disable 状态下生成，与激活 LoRA 对比

COMPARE_Q = "Were Scott Derrickson and Ed Wood from the same country?"

print("===== 激活 LoRA（新代码训练）=====")
model.enable_adapter_layers()
infer(COMPARE_Q)

print()
print("===== 禁用 LoRA（纯 base model）=====")
model.disable_adapter_layers()
infer(COMPARE_Q)

# 恢复 LoRA
model.enable_adapter_layers()